# Phase 2 — SFCN training review

Reviews the SFCN 3D CNN (`bagpipe.models.sfcn`, DESIGN.md §4.2) before
committing to a full production run: image-input EDA, then a small
subject-grouped training run on the real GPU with a **live-updating loss
plot** (train L1 loss + validation MAE, redrawn every epoch).

This is a *review* run, not the leaderboard harness — it trains on a small
subset with a plain random val split (`SFCNRegressor`'s own `val_fraction`),
not the full `evaluate()` grouped 5-fold CV + nested bias-correction CV that
`bag models train-sfcn` runs. That full run is real GPU-hours and is a
separate, deliberate step (see §4 below).

No subject IDs or raw voxel content are printed below — only aggregate
counts, metrics, and de-identified slice visualizations. Outputs are
stripped on commit (nbstripout, see `.pre-commit-config.yaml`) regardless.


In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import clear_output
from sklearn.model_selection import GroupShuffleSplit

from bagpipe.core.config import REPO_ROOT, get_path
from bagpipe.db.export_training_table import export as export_training_table
from bagpipe.models.sfcn import SFCNRegressor
from bagpipe.models.tabular import build_image_matrix

plt.rcParams["figure.dpi"] = 100


## 1. Export + build the image-path matrix

Same `image_paths.parquet` / `globals.parquet` export the other review
notebooks use. `build_image_matrix()` joins CAT12's `mwp1` GM density map
path per session against `globals.parquet` for age, dropping sessions
missing either.


In [ ]:
export_summary = export_training_table()
pd.DataFrame(export_summary).T[["rows"]]


In [ ]:
paths, ages, groups = build_image_matrix(get_path("datasets_dir"))
print(f"{len(paths)} sessions with an mwp1 image + age")
print(f"{len(np.unique(groups))} unique subjects")


## 2. EDA — age distribution, cohort mix, sample slices


In [ ]:
images_df = pd.read_parquet(get_path("datasets_dir") / "image_paths.parquet")
globals_df = pd.read_parquet(get_path("datasets_dir") / "globals.parquet")
eda = images_df.merge(
    globals_df[["subject_key", "session_id", "age"]], on=["subject_key", "session_id"]
).dropna(subset=["age", "image_path_mwp1"])

eda["cohort"].value_counts()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for cohort, sub in eda.groupby("cohort"):
    ax.hist(sub["age"], bins=30, alpha=0.6, label=cohort)
ax.set_xlabel("age at scan")
ax.set_ylabel("sessions")
ax.set_title("Age distribution by cohort (sessions with an SFCN-ready image)")
ax.legend()
plt.show()


### Sample mid-slices across the age range

Sanity check that the images loading into the model are what we expect —
CAT12 `mwp1` GM density maps, MNI-aligned, no obvious corruption. One
subject each from the young/mid/old tail of the age distribution.


In [ ]:
sample = (
    eda.sort_values("age")
    .iloc[[0, len(eda) // 2, len(eda) - 1]]
    .reset_index(drop=True)
)

fig, axes = plt.subplots(len(sample), 3, figsize=(9, 3 * len(sample)))
for row, rec in sample.iterrows():
    volume = nib.load(rec["image_path_mwp1"]).get_fdata(dtype=np.float32)
    mid = np.array(volume.shape) // 2
    slices = [volume[mid[0], :, :], volume[:, mid[1], :], volume[:, :, mid[2]]]
    titles = ["sagittal", "coronal", "axial"]
    for col, (sl, title) in enumerate(zip(slices, titles, strict=True)):
        ax = axes[row, col]
        ax.imshow(np.rot90(sl), cmap="gray")
        ax.set_title(f"age={rec['age']:.0f} — {title}")
        ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Small training run — live loss plot

Subject-grouped subset (`GroupShuffleSplit`, so no subject's sessions leak
between the training pool and the held-out diagnostics split below). Keep
`SUBSET_N` small for an interactive review — scale it up (or point it at
`paths`/`ages`/`groups` directly for the full dataset) once the review looks
right. `SFCNRegressor.fit(..., callback=...)` fires after every epoch; the
callback redraws the plot in place (`clear_output(wait=True)`), so re-running
this cell gives an updating training curve rather than a wall of print
statements.

`BATCH_SIZE` is deliberately small: full-res (113, 137, 113) volumes through
SFCN's 256-channel blocks are memory-hungry, and this is an 8 GB card
already sharing ~1-3 GB with the desktop session — batch 8 OOM'd here, batch
2 didn't. Raise it only if `nvidia-smi` shows enough free memory.


In [ ]:
SUBSET_N = 400  # sessions — small for a fast interactive review, not a real MAE number
EPOCHS = 10
BATCH_SIZE = 2
NUM_WORKERS = 10

rng = np.random.default_rng(0)
subset_idx = rng.choice(len(paths), size=min(SUBSET_N, len(paths)), replace=False)
X_sub, y_sub, groups_sub = paths[subset_idx], ages[subset_idx], groups[subset_idx]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, holdout_idx = next(splitter.split(X_sub, y_sub, groups_sub))
X_train, y_train = X_sub[train_idx], y_sub[train_idx]
X_holdout, y_holdout = X_sub[holdout_idx], y_sub[holdout_idx]
print(f"train pool: {len(X_train)} sessions, holdout (never seen during fit): {len(X_holdout)}")


In [ ]:
history = {"epoch": [], "train_loss": [], "val_mae": []}


def on_epoch_end(epoch, train_loss, val_mae):
    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_mae"].append(val_mae)

    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history["epoch"], history["train_loss"], marker="o", label="train L1 loss")
    if any(v is not None for v in history["val_mae"]):
        ax.plot(history["epoch"], history["val_mae"], marker="o", label="val MAE")
    ax.set_xlabel("epoch")
    ax.set_ylabel("years")
    ax.set_title(f"SFCN training (epoch {epoch + 1}/{EPOCHS})")
    ax.legend()
    plt.show()


regressor = SFCNRegressor(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    val_fraction=0.15,  # internal random split, for the live curve only — see §holdout note above
)
print(f"device: {regressor.device}")
_ = regressor.fit(X_train, y_train, callback=on_epoch_end)


## 4. Holdout diagnostics

Predictions on `X_holdout` — sessions never seen during `fit`, subject-
disjoint from the training pool. With `SUBSET_N` this small and no bias
correction applied, expect a noisy/biased scatter, not leaderboard-quality
numbers; this section is about whether the pipeline behaves sanely (does
error track age reasonably, are there NaNs/outliers), not a final metric.


In [ ]:
preds = regressor.predict(X_holdout)
mae = np.mean(np.abs(preds - y_holdout))
print(f"holdout MAE (uncorrected, {len(y_holdout)} sessions): {mae:.2f} years")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].scatter(y_holdout, preds, s=20, alpha=0.6)
lims = [min(y_holdout.min(), preds.min()), max(y_holdout.max(), preds.max())]
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlabel("true age")
axes[0].set_ylabel("predicted age")
axes[0].set_title("Predicted vs. true age (holdout)")

axes[1].hist(preds - y_holdout, bins=20)
axes[1].axvline(0, color="k", lw=1, ls=":")
axes[1].set_xlabel("predicted - true (years)")
axes[1].set_title("Residuals (holdout)")
plt.tight_layout()
plt.show()


## Next steps

If the loss curve and holdout scatter look sane: scale `SUBSET_N` up (or
drop the subsetting entirely) and hand it to the real harness —
`bag models train-sfcn --config config/models/sfcn.yaml` — which runs the
full subject-grouped 5-fold CV + nested Cole bias-correction CV over all
~5,461 sessions with `image_path_mwp1`, logs to MLflow, and is directly
comparable to the tabular/stacked leaderboard in
`notebooks/stacked_ensemble_review.ipynb`. That's real GPU-hours (multiple
full trainings per outer fold, times the bias-correction inner folds) —
worth agreeing on an epoch/batch-size/time budget before launching it.

Also worth trying before a full run: a pretrained-weights fine-tune
(`model.pretrained_weights_path` in `sfcn.yaml` — UKB-pretrained SFCN
checkpoint, not auto-downloaded) per DESIGN.md §4.2's original plan, which
should converge faster and need less data than training from scratch.
